# Loan Default Prediction - German Credit
Logistic Regression, LDA, QDA + 6 more models. Run the cells top to bottom (all logic lives in `src/`).

## 1. Load & clean
Missing numeric → median, categorical → mode. Target `Risk` (1 = default).

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from IPython.display import display, Image
from src.data import load_raw, prepare, add_features
from src.config import *
raw = load_raw()
print('missing values:', raw.isna().sum().sum())
df = add_features(prepare(raw))
df.head()

## 2. EDA, outliers (IQR capping) and the five chart types

In [ ]:
from src.eda import run_eda
desc, outliers = run_eda(df)
display(desc); display(outliers)

In [ ]:
for f in ['01_pie_risk_distribution','02_bar_default_rate_by_category','03_histograms','04_boxplots_outliers','05_heatmap_correlation','06_scatter_amount_vs_duration','07_zscore_credit_amount']:
    display(Image(str(FIG_DIR / f'{f}.png')))

## 3. Train / test split and 9 models on the top-10 ANOVA-F features

In [ ]:
from sklearn.model_selection import train_test_split
from src.models import get_models
from src.evaluate import fit_and_evaluate, plot_comparison, plot_roc, plot_confusions
X, y = df.drop(columns=[TARGET]), df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
table, fitted, probas = fit_and_evaluate(get_models(), X_tr, X_te, y_tr, y_te, k=TOP_K)
table.round(3)

In [ ]:
baseline = 1 - y_te.mean()
plot_comparison(table, baseline); plot_roc(probas, y_te); plot_confusions(fitted, X_te, y_te)
for f in ['10_model_comparison','11_roc_curves','12_confusion_matrices']:
    display(Image(str(FIG_DIR / f'{f}.png')))
print('Highest accuracy:', table.iloc[0]['Model'], f"{table.iloc[0]['Accuracy']:.1%}", '| baseline', f'{baseline:.1%}')

## 4. Maximum-likelihood coefficients → odds ratios (statsmodels)

In [ ]:
from src.stats_model import odds_ratio_table, plot_odds_ratios
odds, res = odds_ratio_table(fitted['Logistic Regression'], X_tr, y_tr)
plot_odds_ratios(odds)
print(res.summary())
display(odds)
display(Image(str(FIG_DIR / '13_odds_ratios.png')))

## 5. Conclusion
See `docs/PRESENTATION_GUIDE.md` for how to interpret the results. Launch the interactive app with `streamlit run app.py`.